## Import Required Libraries

In [92]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 
import warnings
warnings.filterwarnings('ignore')

import re  
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load Data

In [93]:
# Load the movies_data 
df = pd.read_csv('../data/movies_data.csv')

# Check dataset 
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


## Inspect the dataset 

In [94]:
# Check shape
print("Rows:",df.shape[0])
print("Columns:",df.shape[1])

Rows: 45466
Columns: 24


In [95]:
# Check info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [96]:
# Check columns 
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [97]:
# Check missing values 
df.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [98]:
# Check duplicate values 
df.duplicated().sum()

np.int64(13)

In [99]:
# Drop the duplicated vaues 
df = df.drop_duplicates().reset_index(drop=True)

# Check duplicates 
df.duplicated().sum()

np.int64(0)

**As there are many columns in this dataset in which some are important and some are not. So now take only important columns form this dataset which will help us in recommendation**

In [100]:
# Select important columns 
final_df = df[['title', 'overview', 'genres', 'tagline', 'vote_average', 'popularity']]

# Check df
final_df.head(2)

,title,overview,genres,tagline,vote_average,popularity
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",NaN,7.7,21.946943
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Roll the dice and unleash the excitement!,6.9,17.015539


In [101]:
# Check shape 
print("Rows:",final_df.shape[0])
print("Columns:",final_df.shape[1])

Rows: 45453
Columns: 6


In [102]:
# Check missing values 
final_df.isnull().sum()

title               6
overview          954
genres              0
tagline         25045
vote_average        6
popularity          5
dtype: int64

In [103]:
# Handle missing values
final_df = final_df.dropna(subset=['title'])

final_df['overview'] = final_df['overview'].fillna('')

final_df['tagline'] = final_df['tagline'].fillna('')

# recheck missing values 
final_df.isnull().sum()

title           0
overview        0
genres          0
tagline         0
vote_average    0
popularity      0
dtype: int64

In [104]:
# Check columns values 
for col in final_df.columns:
    print(col)
    print(final_df[col].iloc[0:10])
    print("-"*40)

title
0                      Toy Story
1                        Jumanji
2               Grumpier Old Men
3              Waiting to Exhale
4    Father of the Bride Part II
5                           Heat
6                        Sabrina
7                   Tom and Huck
8                   Sudden Death
9                      GoldenEye
Name: title, dtype: str
----------------------------------------
overview
0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
5    Obsessive master thief, Neil McCauley leads a ...
6    An ugly duckling having undergone a remarkable...
7    A mischievous young boy, Tom Sawyer, witnesses...
8    International action superstar Jean Claude Van...
9    James Bond must unmask the mysterious head of ...
Name: overview, dtype: str
-------------

In [105]:
# Check genres column
final_df['genres'].iloc[0]

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [106]:
# Extract name from genres column
import ast
final_df['genres']= final_df['genres'].apply(lambda x :" ".join( [i['name'] for i in ast.literal_eval(x)]))

In [107]:
# Check dataset 
final_df.head()

,title,overview,genres,tagline,vote_average,popularity
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,,7.7,21.946943
1,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,Roll the dice and unleash the excitement!,6.9,17.015539
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance Comedy,Still Yelling. Still Fighting. Still Ready for...,6.5,11.7129
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,Friends are the people who let you be yourself...,6.1,3.859495
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,Just When His World Is Back To Normal... He's ...,5.7,8.387519


In [108]:
# Make tags column
final_df['tags'] = final_df['overview'] + ' ' + final_df['genres'] + ' ' +  final_df['tagline']

# Check dataset 
final_df.head(2)

,title,overview,genres,tagline,vote_average,popularity,tags
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,,7.7,21.946943,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,Roll the dice and unleash the excitement!,6.9,17.015539,When siblings Judy and Peter discover an encha...


## Text Preprocessing

In [109]:
# Define stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [110]:
# Preprocess the text 
def text_preprocess(text):
    # text in lower case
    text = text.lower() 

    # remove punctuations unnecessary space
    text = re.sub('[^a-zA-Z]',' ',text) 

    # remove unnecessary space
    text = re.sub(r'\s+', ' ', text).strip()

    # split the text 
    text = text.split() 

    # remove stopwords
    text = [word for word in text if word not in stop_words]

    # lemmatize the text 
    text = [lemmatizer.lemmatize(word) for word in text] 

    text = ' '.join(text)
    
    return text

In [111]:
# Apply text_preprocess on tags 
final_df['tags'] = final_df['tags'].apply(text_preprocess)

In [112]:
# Check tags
final_df['tags'].iloc[0:5]

0    led woody andy toy live happily room andy birt...
1    sibling judy peter discover enchanted board ga...
2    family wedding reignites ancient feud next doo...
3    cheated mistreated stepped woman holding breat...
4    george bank recovered daughter wedding receive...
Name: tags, dtype: str

In [128]:
# Make indices for every title 
final_df = final_df.reset_index(drop=True)
indices = pd.Series(final_df.index,index=final_df['title']).drop_duplicates()

# Check indices 
indices

title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Subdue                         45442
Century of Birthing            45443
Betrayal                       45444
Satan Triumphant               45445
Queerama                       45446
Length: 45447, dtype: int64

## Text Vectorization

In [ ]:
# Applying TF-IDF 
vectorizer = TfidfVectorizer(
    max_features=50000, 
    ngram_range=(1,2), 
    sublinear_tf=True
)

# Convert text into tf-idf vectors 
tfidf_matrix = vectorizer.fit_transform(final_df['tags'])

## Calculate Cosine Similarity

In [130]:
# Make a fucntion recommend for calculating cosine similarity
def recommend(title, n=10):

    if title not in indices:
        return ['Movie not found']

    idx = indices[title]

    # If duplicate titles exist
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    sim_score = cosine_similarity(tfidf_matrix[idx],tfidf_matrix).flatten()

    similar_idx = sim_score.argsort()[::-1][1:n+1]

    return df['title'].iloc[similar_idx]

In [135]:
# Check recommend 
recommend("Toy Story")

2996                                           Toy Story 2
15344                                          Toy Story 3
28972                                       The First Star
24512                                    Hawaiian Vacation
10937                                             The Wild
21981                                                 Somm
29189    Mr. Civil Rights: Thurgood Marshall and the NAACP
29369                                 The Pasta Detectives
31493    This Is Not an Exit: The Fictional World of Br...
45063                                                  Cru
Name: title, dtype: str